In [ ]:
import h5py
import pandas as pd
import numpy as np
import os
from matplotlib import pyplot as plt

import math
import warnings
import scipy.optimize
import scipy.stats
import sklearn
from sklearn.svm import LinearSVC
from sklearn.decomposition import PCA
from scipy.stats import wilcoxon
import pathlib

from scipy.spatial import distance
from scipy.linalg import norm

from sklearn.metrics import balanced_accuracy_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.linear_model import LogisticRegression

# %matplotlib tk

In [ ]:
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import decoding_utils as du

In [ ]:
#Paths to all of the useful supplemental tables and tensors
active_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor.hdf5"
passive_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor_passive.hdf5"
opto_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbn_opto_tensor_unit_grouped.hdf5"

stim_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_stim_table_no_filter.csv"
unit_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_units_with_responsiveness.csv"
unit_table_with_rf_stats = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/units_with_rf_stats.csv"
unit_table_opto_metrics = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/unit_opto_metrics.csv"
unit_cluster_labels = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/unit_cluster_labels.csv"
unit_vis_responsive = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/units_with_vis_responsiveness.csv"

sessions_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/vbn_s3_cache/visual-behavior-neuropixels-0.4.0/project_metadata/ecephys_sessions.csv"

In [ ]:
stim_table = pd.read_csv(stim_table_file)
unit_table = pd.read_csv(unit_table_file)

In [ ]:
unitData = h5py.File(active_tensor_file)

In [ ]:
sessions = pd.read_csv(sessions_table_file)

In [ ]:
from notebook_utils import (
    make_mono_colormap,
    mean_paired_image_mat_from_stim_table,
    get_contingent_engaged_trials,
    beh_mat_from_stim_table,
    get_omission_response_rate,
    get_post_omission_response_rate,
    get_shared_hit_rate,
    get_private_hit_rate,
    get_shared_fa_rate,
    get_private_fa_rate,
    get_shared_nonchange_response_rate,
    get_private_nonchange_response_rate,
    mean_beh_mat_across_sessions,
    skip_diag_masking,
    calcDprime,
    calcHitRate,
    get_session_engaged_dprime,
    get_session_engaged_hit_count,
    get_experience_session_id_for_mouse,
    get_omission_mean,
    get_post_omission_mean,
    get_shared_change_mean,
    get_private_change_mean,
    get_shared_catch_mean,
    get_private_catch_mean,
    get_shared_nonchange_mean,
    get_private_nonchange_mean,
    multiple_comparisons,
    comparison_matrix,
    plot_image,
    get_session_stim_table_stats
)

In [ ]:
save_dir = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/flash_decoding_metrics"

In [ ]:
regions = ('VISall', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'LGd', 'LP', 'SCMRN', 'Hipp')

## To generate flash metrics (skip to load pre-computed flash metrics)

In [ ]:
def decodeSingleClass(sp, y, nCrossVal, model, unitSampleSize):
    '''
    sp: units x trials
    y: labels 
    nCrossVal: number of cross validation splits to run
    '''
    nUnits = sp.shape[0]
    warnings.filterwarnings('ignore')
    summary = {sampleSize: 
                {metric: [] for metric in  ('trainAccuracy',
                                            'featureWeights',
                                            'accuracy',
                                            'prediction',
                                            'confidence', 
                                            'balanced_accuracy')}
                for sampleSize in unitSampleSize}
    
    for sampleSize in unitSampleSize:
        if nUnits < sampleSize:
            continue
        if sampleSize>1:
            if sampleSize==nUnits:
                nSamples = 1
                unitSamples = [np.arange(nUnits)]
            else:
                # >99% chance each neuron is chosen at least once
                nSamples = int(math.ceil(math.log(0.01)/math.log(1-sampleSize/nUnits)))
                unitSamples = [np.random.choice(nUnits,sampleSize,replace=False) for _ in range(nSamples)]
        elif sampleSize==1:
            nSamples = nUnits
            unitSamples = [[i] for i in range(nUnits)]
        else: #if sampleSize<1, just run 1 iteration with all the units available
            nSamples = 1
            unitSamples = [np.arange(nUnits)]

        for metric in summary[sampleSize]:
            summary[sampleSize][metric].append([])

        for unitSamp in unitSamples:
            cv = du.trainDecoder(model,sp[unitSamp].T,y,nCrossVal)
            summary[sampleSize]['trainAccuracy'][-1].append(np.mean(cv['train_score']))
            summary[sampleSize]['featureWeights'][-1].append(np.mean(cv['coef'],axis=0).squeeze())
            summary[sampleSize]['accuracy'][-1].append(np.mean(cv['test_score']))
            summary[sampleSize]['prediction'][-1].append(cv['predict'])
            summary[sampleSize]['confidence'][-1].append(cv['decision_function'])
            summary[sampleSize]['balanced_accuracy'][-1].append(sklearn.metrics.balanced_accuracy_score(y.astype(bool), cv['predict'].astype(bool)))

        for metric in summary[sampleSize]:
            if metric == 'prediction':
                summary[sampleSize][metric][-1] = scipy.stats.mode(summary[sampleSize][metric][-1],axis=0)[0][0]
            elif metric != 'featureWeights':
                summary[sampleSize][metric][-1] = np.median(summary[sampleSize][metric][-1],axis=0)
    
    warnings.filterwarnings('default')
    return summary

In [ ]:
unitSampleSize = [20,40,-1]
unitSampleNames = [u if u>=1 else 'all' for u in unitSampleSize]
respWin = slice(20,100)

for isess, session in sessions.iterrows():
    # if isess%10==0:
    print(isess)

    sessionId = session['ecephys_session_id']

    stim = stim_table[(stim_table['session_id']==sessionId) & stim_table['active']].reset_index()
    im_ids = stim['image_name'].values
    previous_im_ids = np.insert(stim['image_name'].values, 0, 'omitted')[:-1]
    previous_im_ids_skipping_omitted = []
    for ind, id in enumerate(previous_im_ids):
        if not id=='omitted':
            previous_im_ids_skipping_omitted.append(id)
        else:
            if ind==0:
                previous_im_ids_skipping_omitted.append(np.nan)
            else:
                previous_im_ids_skipping_omitted.append(previous_im_ids[ind-1])

    previous_im_ids_skipping_omitted = np.array(previous_im_ids_skipping_omitted)

    stim['previous_image_skipping_omitted'] = previous_im_ids_skipping_omitted
    unique_images = [im for im in np.unique(stim['image_name'].values) if im not in ['im083_r', 'im111_r', 'omitted']] + ['im083_r', 'im111_r']

    units = unit_table.set_index('unit_id').loc[unitData[str(sessionId)]['unitIds'][:]]
    spikes = unitData[str(sessionId)]['spikes']
    highQuality = du.apply_unit_quality_filter(units, no_abnorm=False)

    ### IN LOOP ###
    for region in regions:
        inRegion = du.getUnitsInRegion(units,region)
        final_unit_filter = highQuality & inRegion
        
        cols_to_add = [f'{metric}_{region}_{unitSampleName}' for unitSampleName in unitSampleNames for metric in ['previous_image_confidence', 'change_confidence']]
        stim[cols_to_add] = np.nan
        
        if np.sum(final_unit_filter)<10:
            continue

        sp = np.zeros((final_unit_filter.sum(),spikes.shape[1],spikes.shape[2]),dtype=bool)
        for i,u in enumerate(np.where(final_unit_filter)[0]):
            sp[i]=spikes[u,:,:]

        flash_response = sp[:, :, respWin].mean(axis=2)

        image_decoder_summary = {}
        for im in unique_images:
            y = im_ids == im
            model = LinearSVC(C=1.0,max_iter=int(1e4),class_weight='balanced')
            results = decodeSingleClass(flash_response, y, 5, model, unitSampleSize)
            image_decoder_summary[im] = results

        model = LinearSVC(C=1.0,max_iter=int(1e4), class_weight='balanced')
        change_results = decodeSingleClass(flash_response, stim['is_change'].values, 5, model, unitSampleSize)

        for unitSample, unitSampleName in zip(unitSampleSize, unitSampleNames):
            for i, row in stim.iterrows():
                previous_stim_id = row['previous_image_skipping_omitted']
                if previous_stim_id in image_decoder_summary:
                    previous_results = image_decoder_summary[previous_stim_id]
                    if len(previous_results[unitSample]['confidence'])>0:
                        confidence = previous_results[unitSample]['confidence'][0][i]
                        stim.at[i, f'previous_image_confidence_{region}_{unitSampleName}'] = confidence

            if len(change_results[unitSample]['confidence'])>0:
                stim[f'change_confidence_{region}_{unitSampleName}'] = change_results[unitSample]['confidence'][0]

    cols_to_save = ['session_id', 'is_change', 'previous_image_skipping_omitted', 'image_name'] + \
                    [f'previous_image_confidence_{region}_{unitSampleName}' for region in regions for unitSampleName in unitSampleNames] + \
                    [f'change_confidence_{region}_{unitSampleName}' for region in regions for unitSampleName in unitSampleNames]
    stim[cols_to_save].to_csv(os.path.join(save_dir, f'{sessionId}_responseWin_20to100.csv'), index=False)

In [ ]:
save_dir = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/flash_decoding_metrics"

#stim_files = [os.path.join(save_dir, f) for f in os.listdir(save_dir) if 'previous' not in f]
stim_files = [os.path.join(save_dir, f) for f in os.listdir(save_dir) if 'responseWin_20to100' in f]
dfs = []
for stim_file in stim_files:
    df = pd.read_csv(stim_file)
    stim = stim_table[stim_table['session_id']==int(os.path.basename(stim_file).split('.')[0].split('_')[0])]
    df = df.merge(stim.reset_index(), left_index=True, right_index=True, suffixes=('', '_stimtable'))     
    
    dfs.append(df)

stimtable_with_flash_metrics = pd.concat(dfs)

In [ ]:
stimtable_with_flash_metrics = stimtable_with_flash_metrics.drop(columns=[c for c in stimtable_with_flash_metrics if 'Unnamed' in c])
stimtable_with_flash_metrics.head()

## Load precomputed flash metrics (if you didn't compute them fresh above)

In [ ]:
def trainModel(model,X,y,nSplits):
    classVals = np.unique(y)
    nClasses = len(classVals)
    nSamples = len(y)
    cv = {'estimator': [sklearn.base.clone(model) for _ in range(nSplits)]}
    cv['train_balanced_accuracy'] = []
    cv['test_balanced_accuracy'] = []
    cv['predict'] = np.full(nSamples, '', dtype='O')
    cv['predict_proba'] = np.full((nSamples,nClasses),np.nan)
    cv['coef'] = []
    modelMethods = dir(model)
    trainInd,testInd = du.getTrainTestSplits(y,nSplits,hasClasses=False)
    for num_iter, (estimator,train,test) in enumerate(zip(cv['estimator'],trainInd,testInd)):
        estimator.fit(X[train],y[train])
        cv['train_balanced_accuracy'].append(balanced_accuracy_score(y[train], estimator.predict(X[train])))
        cv['test_balanced_accuracy'].append(balanced_accuracy_score(y[test], estimator.predict(X[test])))
        cv['predict'][test] = estimator.predict(X[test])
        for method in ('predict_proba',):
            if method in modelMethods:
                cv[method][test] = getattr(estimator,method)(X[test])
        for attr in ('coef_',):
            if attr in estimator.__dict__:
                cv[attr[:-1]].append(getattr(estimator,attr))
    return cv

In [ ]:
session_model_results = {s:{} for s in stimtable_with_flash_metrics['session_id'].unique()}
areas_to_run = ('VISall', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'LGd', 'LP', 'SCMRN', 'Hipp')
#metrics_to_run = ['cosine_distance_from_previous_flash_in_lda_space', 'lda_on_change']
metrics_to_run = ['previous_image_confidence', 'change_confidence']
samplesizes = ['20', '40', 'all']

columns_to_predict = ['is_change', 'lickbout_for_flash_during_response_window']
filters = [lambda x: [True]*len(x), lambda x: ~x['is_change']]

warnings.filterwarnings("ignore")
for isess, session in enumerate(stimtable_with_flash_metrics['session_id'].unique()):
    if isess%10==0:
        print(isess)
    stim = stimtable_with_flash_metrics[stimtable_with_flash_metrics['session_id']==session]
    stim = stim[stim['engaged'] & \
                stim['no_abnorm'] & \
                ~stim['grace_period_after_hit']]# & \
                #~stim['omitted'] & \
                #~stim['previous_omitted'] & \
                #~(stim['is_change'] | stim['is_sham_change'] | stim['is_prechange']) & \
                #~(stim['is_change'] | stim['is_sham_change']) & \
                #~(stim['is_change'])]
    
    for samplesize in samplesizes:
        for area in areas_to_run:
            for metrics in metrics_to_run:
                
                if isinstance(metrics, str):
                        metrics = metrics + '_' + area + '_' + samplesize if not 'lick' in metrics else metrics
                        metrics_name = metrics
                else:
                    metrics = [m+'_'+area if not 'lick' in m else m for m in metrics]
                    metrics_name = 'combo'
                    for m in metrics_to_run:
                        metrics_name = metrics_name + '__' + m
                
                for column_to_predict, filter in zip(columns_to_predict, filters):
                    
                    curated_table = stim[filter(stim)]
                    if len(curated_table)==0:
                        continue

                    X = curated_table[metrics].to_numpy().astype(float)
                    no_nan_inds = [irow for irow, row in enumerate(X) if not np.any(np.isnan(row))]
                    if len(no_nan_inds)==0:
                        session_model_results[session].update({'train_balanced_accuracy' + '_' + metrics_name: np.nan, 
                                                                'test_balanced_accuracy' + '_' + metrics_name: np.nan})
                        continue
                    #X_nonan = X[no_nan_inds]
                    X_nonan = X[no_nan_inds].reshape(-1,1)

                    X_nonan = (X_nonan - X_nonan.mean(axis=0))/X_nonan.std(axis=0)
                    y = curated_table[column_to_predict].values
                    y_nonan = y[no_nan_inds]
                    model = LogisticRegression(random_state=0, solver='liblinear', class_weight='balanced')
                    res = trainModel(model, X_nonan, y_nonan, 5,)

                    session_model_results[session].update({f'{column_to_predict}_train_balanced_accuracy_{metrics_name}': np.mean(res['train_balanced_accuracy']),
                                                            f'{column_to_predict}_test_balanced_accuracy_{metrics_name}': np.mean(res['test_balanced_accuracy'])})

In [ ]:
sess_change_df = pd.DataFrame.from_dict(session_model_results, orient='index')

In [ ]:
sess_change_df = sess_change_df.merge(sessions, left_index=True, right_on='ecephys_session_id').set_index('ecephys_session_id')

In [ ]:
sess_change_df = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/session_logreg_previmage_vs_change_responseWindow_20to100.csv", index_col=0)

In [ ]:
from analysis_utils import multiple_comparisons, comparison_matrix

In [ ]:
from statsmodels.stats.multitest import multipletests
def multiple_comparisons(pvalues):
    if isinstance(pvalues, dict):
        old_values = list(pvalues.values())
    else:
        old_values = pvalues
    
    reject, corrected_pvalues, _, _ = multipletests(old_values, alpha=0.05, method='fdr_bh')

    if isinstance(pvalues, dict):
        return {list(pvalues.keys())[i]: corrected_pvalues[i] for i in range(len(pvalues))}

    return reject, corrected_pvalues


def comparison_matrix(*values, test_func=scipy.stats.wilcoxon):

    pvalue_matrix = np.full((len(values), len(values)), np.nan)
    for ind1, vals1 in enumerate(values):
        for ind2, vals2 in enumerate(values):
            if ind1==ind2:
                continue
            
            p = test_func(vals1, vals2, nan_policy='omit')
            pvalue_matrix[ind1, ind2] = p.pvalue
    
    diag_mask = np.ones(pvalue_matrix.shape, dtype=bool)
    diag_mask[np.diag_indices_from(diag_mask)] = False
    corrected_pvals = multipletests(pvalue_matrix[np.where(diag_mask)], method='fdr_bh')

    pvalue_matrix[np.where(diag_mask)] = corrected_pvals[1]
    sig_matrix = pvalue_matrix<0.05


    return pvalue_matrix, sig_matrix


In [ ]:
from vbn_utils import formatFigure

In [ ]:
areas_to_run = ('VISall', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'LGd', 'LP', 'SCMRN', 'Hipp')

cols = ['is_change', 'lickbout_for_flash_during_response_window']
samplesize = 'all'
area_pvalues_change_vs_image = {}
area_change_conf_values = {c:{} for c in cols}
area_previous_image_conf_values = {c:{} for c in cols}
for col in cols:
    fig, ax = plt.subplots()
    fig.suptitle(col)
    for ia, area in enumerate(areas_to_run):
        previous_image_conf = sess_change_df[col+'_test_balanced_accuracy_previous_image_confidence_' + area + '_' + samplesize].agg(['mean', 'sem'])
        change_conf = sess_change_df[col+'_test_balanced_accuracy_change_confidence_' + area + '_' + samplesize].agg(['mean', 'sem'])

        previous_image_conf_values = sess_change_df[col+'_test_balanced_accuracy_previous_image_confidence_' + area + '_' + samplesize].dropna()
        change_conf_values = sess_change_df[col+'_test_balanced_accuracy_change_confidence_' + area + '_' + samplesize].dropna()
        area_change_conf_values[col][area] = change_conf_values
        area_previous_image_conf_values[col][area] = previous_image_conf_values
        area_pvalues_change_vs_image[area] = wilcoxon(previous_image_conf_values, change_conf_values, nan_policy='omit')[1]

        # ax.violinplot(changelda, [ia])
        ax.plot(ia, previous_image_conf['mean'], 'ko')
        ax.errorbar(ia, previous_image_conf['mean'], previous_image_conf['sem'], color='k')
        
        ax.plot(ia, change_conf['mean'], 'ro')
        ax.errorbar(ia, change_conf['mean'], change_conf['sem'], color='r')

    ax.set_xticks(np.arange(len(areas_to_run)))
    ax.set_xticklabels(areas_to_run)
    ax.set_ylabel(f'balanced accuracy')
    ax.legend(['previous_image_confidence', 'change_confidence'])

    #Correct for multiple comparisons
    area_pvalues_change_vs_image = multiple_comparisons(area_pvalues_change_vs_image)

    maxy = ax.get_ylim()[1]
    for ia, area in enumerate(areas_to_run):
        if area_pvalues_change_vs_image[area]<0.05:
            ax.text(ia, maxy, '*', horizontalalignment='center', verticalalignment='center')
    
    #formatFigure(fig, ax, saveName=os.path.join(save_dir, f'{col}_{samplesize}_neurons.pdf'))

In [ ]:
for col in cols:
    for conf_values, name in zip([area_change_conf_values[col], area_previous_image_conf_values[col]], ['change confidence', 'image confidence']):
        ps,sigs = comparison_matrix(*list(conf_values.values()), test_func=scipy.stats.ranksums)
        fig, ax = plt.subplots()
        fig.suptitle(name)
        # Create a mask for coordinates that are < 0.05
        mask = ps < 0.05
        # Plot the modified array
        plt.xticks(range(len(ps)), list(conf_values.keys()), rotation=90)
        plt.yticks(range(len(ps)), list(conf_values.keys()))
        # Draw red asterisks at coordinates where ps < 0.05
        ax.plot(*np.where(mask), 'w*')
        im = ax.imshow(ps, clim=(0,1))
        plt.xlabel('Area')
        plt.ylabel('Area')
        plt.colorbar(im, label='corrected p-value')
        #formatFigure(fig, ax, saveName=os.path.join(save_dir, f'{col}_{name}_pvalue_comparison_matrix.pdf'))


In [ ]:
experience_level = ['Familiar', 'Novel']
cols = ['is_change', 'lickbout_for_flash_during_response_window']
samplesize = 'all'
for experience in experience_level:
    for col in cols:
        fig, ax = plt.subplots()
        fig.suptitle(f'{col} {experience}')
        for ia, area in enumerate(areas_to_run):
            df_to_use = sess_change_df[sess_change_df['experience_level']==experience]
            previous_image_conf = df_to_use[col+'_test_balanced_accuracy_previous_image_confidence_' + area + '_' + samplesize].agg(['mean', 'sem'])
            change_conf = df_to_use[col+'_test_balanced_accuracy_change_confidence_' + area + '_' + samplesize].agg(['mean', 'sem'])

            # ax.violinplot(changelda, [ia])
            ax.plot(ia, previous_image_conf['mean'], 'ko')
            ax.errorbar(ia, previous_image_conf['mean'], previous_image_conf['sem'], color='k')
            
            ax.plot(ia, change_conf['mean'], 'ro')
            ax.errorbar(ia, change_conf['mean'], change_conf['sem'], color='r')

        ax.set_xticks(np.arange(len(areas_to_run)))
        ax.set_xticklabels(areas_to_run)
        ax.set_ylabel(f'balanced accuracy')
        ax.legend(['previous_image_confidence', 'change_confidence'])

In [ ]:
# image_name_dict is also available in notebook_utils
image_name_dict = {
    'H': ['im005_r', 'im024_r', 'im034_r', 'im087_r', 'im104_r','im114_r','im083_r','im111_r'],
    'G': ['im012_r', 'im036_r', 'im044_r','im047_r','im078_r','im115_r','im083_r', 'im111_r']
}

In [ ]:
plt.rcParams['font.size'] = 14
area = 'VISam'
samplesize = 'all'
metrics_to_plot = ['previous_image_confidence', 'change_confidence']
colors = [(1,0,1), (0, 1, 1)]
for metric_name, sign, color in zip(metrics_to_plot, [-1,1], colors):
    cmap = make_mono_colormap((1,1,1), color).reversed()
    #metric_name = metric_name +'_'+area if metric_name != 'lickbout_for_flash_during_response_window' else metric_name
    metric_name = f'{metric_name}_{area}_{samplesize}' if metric_name != 'lickbout_for_flash_during_response_window' else metric_name

    fig, ax = plt.subplots()
    fig.suptitle(f'{metric_name}')
    images, counts, vals = mean_paired_image_mat_from_stim_table(stimtable_with_flash_metrics, metric_name, experience='Novel', image_set='H')
    vals = vals*sign
    vals_normed = (vals-vals.min())/np.max(vals-vals.min())
    im = ax.imshow(vals_normed, cmap='inferno')
    images = [im.replace('_r', '') for im in images]  # Remove '_r' from image names for better readability
    ax.set_xticks(np.arange(len(images)))
    ax.set_xticklabels(images, rotation=90)
    ax.set_yticks(np.arange(len(images)))
    ax.set_yticklabels(images)
    plt.colorbar(im)
    images, counts, response_vals = mean_paired_image_mat_from_stim_table(stimtable_with_flash_metrics, 'lickbout_for_flash_during_response_window', experience='Novel', image_set='H')
    images = [im.replace('_r', '') for im in images]  # Remove '_r' from image names for better readability
    fig, ax = plt.subplots()
    fig.suptitle(metric_name)
    ax.plot(response_vals.flatten(), vals.flatten(), 'k.')

    # nodiag = response_vals>0.4
    nodiag = ~np.eye(response_vals.shape[0], dtype=bool)

    print(metric_name, np.corrcoef(response_vals[nodiag].flatten(), vals[nodiag].flatten())[0,1]**2)


fig, ax = plt.subplots()
fig.suptitle('licks')
im = ax.imshow(response_vals, cmap='inferno', clim=(0,1))
ax.set_xticks(np.arange(len(images)))
ax.set_xticklabels(images, rotation=90)
ax.set_yticks(np.arange(len(images)))
ax.set_yticklabels(images)
plt.colorbar(im)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
area = 'VISam'
samplesize = 'all'
metrics_to_plot = ['previous_image_confidence', 'change_confidence']

experience='Novel'
image_set='H'

sessions = stimtable_with_flash_metrics[(stimtable_with_flash_metrics['experience_level']==experience)&(stimtable_with_flash_metrics['image_set']==image_set)]['session_id'].unique()

colors = [(1,0,1), (0, 1, 1)]
metric_lick_correlations = {}
metric_mats = {}
for session in sessions: 
    for metric_name, sign, color in zip(metrics_to_plot, [-1,1], colors):
        cmap = make_mono_colormap((1,1,1), color).reversed()
        #metric_name = metric_name +'_'+area if metric_name != 'lickbout_for_flash_during_response_window' else metric_name
        metric_name = f'{metric_name}_{area}_{samplesize}' if metric_name != 'lickbout_for_flash_during_response_window' else metric_name
        if metric_name not in metric_lick_correlations:
            metric_lick_correlations[metric_name] = []
            metric_mats[metric_name] = []

        fig, ax = plt.subplots()
        fig.suptitle(f'{metric_name}')
        images, counts, vals = mean_paired_image_mat_from_stim_table(stimtable_with_flash_metrics, metric_name, experience='Novel', image_set='H', session_id=session)
        vals = vals*sign
        vals_normed = (vals-vals.min())/np.max(vals-vals.min())
        metric_mats[metric_name].append(vals_normed)
        
        im = ax.imshow(vals_normed, cmap='inferno')
        plt.colorbar(im)
        images, counts, response_vals = mean_paired_image_mat_from_stim_table(stimtable_with_flash_metrics, 'lickbout_for_flash_during_response_window', experience='Novel', image_set='H', session_id= session)
        fig, ax = plt.subplots()
        fig.suptitle(metric_name)
        ax.plot(response_vals.flatten(), vals.flatten(), 'k.')

        nodiag = ~np.eye(response_vals.shape[0], dtype=bool)

        print(metric_name, np.corrcoef(response_vals[nodiag].flatten(), vals[nodiag].flatten())[0,1]**2)
        metric_lick_correlations[metric_name].append(np.corrcoef(response_vals[nodiag].flatten(), vals[nodiag].flatten())[0,1]**2)


fig, ax = plt.subplots()
fig.suptitle('licks')
im = ax.imshow(response_vals, cmap='inferno', clim=(0,1))
plt.colorbar(im)

In [ ]:
for metric in metric_mats:
    mean_across_sessions = np.nanmean(metric_mats[metric], axis=0)
    fig, ax = plt.subplots()
    fig.suptitle(metric)
    im = ax.imshow(mean_across_sessions, cmap='inferno', clim=(0,1))
    plt.colorbar(im)

In [ ]:
fig, ax = plt.subplots()
fig.set_size_inches(6,6)
ax.plot(metric_lick_correlations['previous_image_confidence_VISam_all'], metric_lick_correlations['change_confidence_VISam_all'], 'ko', alpha=0.5)
ax.set_aspect('equal')
ax.plot([0,1], [0,1], 'k--')
ax.set_xlim(-0.05,0.2)

## Lick rates for holdovers during familiar and novel sessions

In [ ]:
sessions_table = pd.read_csv(sessions_table_file)
good_sessions = sessions_table[sessions_table['abnormal_histology'].isnull() & sessions_table['abnormal_activity'].isnull()]
sessions_to_analyze = good_sessions['ecephys_session_id'].unique()

session_beh_mats = {}
beh_mat_array = []
count_array = []
session_labels = []
experience_labels = []
image_set_labels = []
for _, session in good_sessions.iterrows():
    session_id = session['ecephys_session_id']
    mouse_id = session['mouse_id']
    experience = session['experience_level']

    ims, counts, beh_mat = beh_mat_from_stim_table(stim_table, session_id)
    image_set = 'G' if 'im036_r' in ims else 'H'
    
    beh_mat_array.append(beh_mat)
    count_array.append(counts)
    session_labels.append(session_id)
    experience_labels.append(experience)
    image_set_labels.append(image_set)


    session_beh_mats[session_id] = {'beh_mat': beh_mat, 'counts': counts, 'images': ims}
    

In [ ]:
mouse_session_counts = good_sessions.value_counts('mouse_id')
mice_with_fam_and_nov_sessions = mouse_session_counts[mouse_session_counts > 1].index.values
mice_with_fam_and_nov_sessions


In [ ]:
im_83_hit_rates = {'Familiar':[], 'Novel':[], 'Familiar_from_holdover':[], 'Novel_from_holdover':[], }
im_111_hit_rates = {'Familiar':[], 'Novel':[], 'Familiar_from_holdover':[], 'Novel_from_holdover':[], }

for mouse in mice_with_fam_and_nov_sessions:

    for experience in ['Familiar', 'Novel']:

        session_id = get_experience_session_id_for_mouse(sessions_table, mouse, experience)

        beh_mat = session_beh_mats[session_id]['beh_mat']
        counts = session_beh_mats[session_id]['counts']
        ims = session_beh_mats[session_id]['images']

        beh_mat_no_diag = skip_diag_masking(beh_mat)
        counts_no_diag = skip_diag_masking(counts)


        for ind, imrates in zip([6,7], [im_83_hit_rates[experience], im_111_hit_rates[experience]]):

            ind_count = np.nansum(counts_no_diag[:, ind])
            if (ind_count > 10) & (get_session_engaged_dprime(stim_table, session_id)>1) & (get_session_engaged_hit_count(stim_table, session_id)>50):

                #print(ind_count, get_session_engaged_dprime(stim_table, session_id), get_session_engaged_hit_count(stim_table, session_id))
                imrates.append(np.nanmean(beh_mat_no_diag[:, ind]))
            
            else:
                imrates.append(np.nan)

In [ ]:
import vbn_utils

In [ ]:
fig, ax = plt.subplots()
ax.plot(im_83_hit_rates['Familiar'], im_83_hit_rates['Novel'], 'ko')
ax.plot(im_111_hit_rates['Familiar'], im_111_hit_rates['Novel'], 'ko', mfc='w')
ax.set_aspect('equal')
# ax.legend(['im083', 'im111'], frameon=False)
ax.plot([0,1], [0,1], 'k--')
ax.set_xlabel('Hit rate during Familiar session')
ax.set_ylabel('Hit rate during Novel session')

vbn_utils.formatFigure(fig, ax, )

In [ ]:
#Add dprime and hit count to sessions table
dprimes = []
hit_counts = []
for _, session in sessions.iterrows():

    session_id = session['ecephys_session_id']
    dprime = get_session_engaged_dprime(stim_table, session_id)
    hit_count = get_session_engaged_hit_count(stim_table, session_id)

    dprimes.append(dprime)
    hit_counts.append(hit_count)


sessions['engaged_dprime'] = dprimes
sessions['engaged_hitcount'] = hit_counts

In [ ]:
good_sessions = sessions[sessions['abnormal_histology'].isnull() & sessions['abnormal_activity'].isnull()]
good_behavior_sessions = good_sessions[(good_sessions['engaged_dprime']>=1)&(good_sessions['engaged_hitcount']>=50)]

In [ ]:
areas_to_run = ('VISall', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'LGd', 'LP', 'SCMRN')
metrics_to_run = ['previous_image_confidence', 'change_confidence']
samplesizes = ['20', '40', 'all']

response_rate_summary = {s:{f'{image_set}_{experience_level}_{metric}_{area}_{samplesize}_{response_type}': np.nan \
                            for image_set in ['G', 'H'] \
                            for experience_level in ['Familiar', 'Novel'] \
                            for metric in ['previous_image_confidence', 'change_confidence'] \
                            for area in areas_to_run \
                            for samplesize in samplesizes \
                            for response_type in ['private_nonchange', 'shared_nonchange', 'private_hit', 'shared_hit', 'private_fa', 'shared_fa', 'omission', 'postomission']}
                        for s in good_behavior_sessions['ecephys_session_id'].values}

for isess, session in good_behavior_sessions.iterrows():
    print(isess)
    experience_level = session['experience_level']
    image_set = session['image_set']
    session_id = session['ecephys_session_id']
    session_stim_table = stimtable_with_flash_metrics[stimtable_with_flash_metrics['session_id']==session_id]
    
    for metric in metrics_to_run:
        for area in areas_to_run:
            for samplesize in samplesizes:
                metric_name = f'{metric}_{area}_{samplesize}'
                response_rate_summary[session_id][image_set + '_' + experience_level + '_' + metric_name + '_private_nonchange'] = get_private_nonchange_mean(session_stim_table, metric_name)
                response_rate_summary[session_id][image_set + '_' + experience_level + '_' + metric_name + '_private_hit'] = get_private_change_mean(session_stim_table, metric_name)
                response_rate_summary[session_id][image_set + '_' + experience_level + '_' + metric_name + '_private_fa'] = get_private_catch_mean(session_stim_table, metric_name)
                response_rate_summary[session_id][image_set + '_' + experience_level + '_' + metric_name + '_shared_nonchange'] = get_shared_nonchange_mean(session_stim_table, metric_name)
                response_rate_summary[session_id][image_set + '_' + experience_level + '_' + metric_name + '_shared_hit'] = get_shared_change_mean(session_stim_table, metric_name)
                response_rate_summary[session_id][image_set + '_' + experience_level + '_' + metric_name + '_shared_fa'] = get_shared_catch_mean(session_stim_table, metric_name)
                response_rate_summary[session_id][image_set + '_' + experience_level + '_' + metric_name + '_omission'] = get_omission_mean(session_stim_table, metric_name)
                response_rate_summary[session_id][image_set + '_' + experience_level + '_' + metric_name + '_postomission'] = get_post_omission_mean(session_stim_table, metric_name)

In [ ]:
response_rate_df = pd.DataFrame.from_dict(response_rate_summary, orient='index')

In [ ]:
response_rate_df.to_csv(os.path.join(save_dir, 'previous_image_and_change_confidence_response_metrics.csv'))

In [ ]:
aggregated_labels = ['Familiar_shared', 'Familiar_private', 'Novel_shared', 'Novel_private', 'Novel', 'Familiar']
response_categories = ['nonchange', 'hit', 'fa', 'omission', 'postomission']
samplesize = 'all'
aggregations = (
                ['Familiar', 'private', 'nonchange'],
                ['Familiar', 'shared', 'nonchange'],
                ['Novel', 'private', 'nonchange'],
                ['Novel', 'shared', 'nonchange'],
                ['Familiar', 'private', 'hit'],
                ['Familiar', 'shared', 'hit'],
                ['Novel', 'private', 'hit'],
                ['Novel', 'shared', 'hit'],
                ['Familiar', 'private', 'fa'],
                ['Familiar', 'shared', 'fa'],
                ['Novel', 'private', 'fa'],
                ['Novel', 'shared', 'fa'],
                ['Familiar', '', 'omission'],
                ['Novel', '', 'omission'],
                ['Familiar', '', 'postomission'],
                ['Novel', '', 'postomission'],
                )



cols_to_aggregate = []
for metric in ['previous_image_confidence', 'change_confidence']:
    for area in areas_to_run:
        for aggregation in aggregations:
                cols = [c for c in response_rate_df.columns if \
                        aggregation[0] in c and \
                        aggregation[1] in c and \
                        f'_{aggregation[2]}' in c and \
                        metric in c and f'_{area}_' in c and f'_{samplesize}_' in c]
                if len(cols)>0:
                    cols_to_aggregate.append(cols)

In [ ]:
labels = []
values = []
means = []
sems = []
for cols in cols_to_aggregate:
    labels.append(cols[0][2:])
    vals = response_rate_df[cols].to_numpy().flatten()
    vals = vals[~np.isnan(vals)]

    values.append(vals)
    means.append(np.mean(vals))
    sems.append(np.std(vals)/len(vals)**0.5)

labels = np.array(labels)
values = np.array(values)
means = np.array(means)
sems = np.array(sems)

In [ ]:
for area in areas_to_run:
    for metric, sign in zip(['previous_image_confidence', 'change_confidence'], [-1,1]):
        fig, ax = plt.subplots()
        fig.suptitle(metric + ' ' + area)
        #fig.set_size_inches([8, 14])
        inds = [i for i, label in enumerate(labels) if metric in label and f'_{area}_' in label]
        metric_labels = [l.replace(f'_{metric}', '').replace(f'_{area}', '').replace(f'_{samplesize}', '') for l in labels[inds]]
        metric_values = values[inds] * sign
        ax.violinplot(metric_values, showmedians=True)
        # ax.plot(means, 'ko')
        # ax.errorbar(np.arange(len(means)), means, yerr=sems, color='k')
        ax.set_xticks(np.arange(len(metric_labels))+1)
        ax.set_xticklabels(metric_labels, rotation=90)
plt.tight_layout()

## Plot flash responses in PCA space to illustrate different strategies

In [ ]:
sessionId = good_behavior_sessions.iloc[0]['ecephys_session_id']#session['ecephys_session_id']
region = 'VISall'

stim = stim_table[(stim_table['session_id']==sessionId) & stim_table['active']].reset_index()
im_ids = stim['image_name'].values
previous_im_ids = np.insert(stim['image_name'].values, 0, 'omitted')[:-1]
previous_im_ids_skipping_omitted = []
for ind, id in enumerate(previous_im_ids):
    if not id=='omitted':
        previous_im_ids_skipping_omitted.append(id)
    else:
        if ind==0:
            previous_im_ids_skipping_omitted.append(np.nan)
        else:
            previous_im_ids_skipping_omitted.append(previous_im_ids[ind-1])

previous_im_ids_skipping_omitted = np.array(previous_im_ids_skipping_omitted)

stim['previous_image_skipping_omitted'] = previous_im_ids_skipping_omitted
unique_images = [im for im in np.unique(stim['image_name'].values) if im not in ['im083_r', 'im111_r', 'omitted']] + ['im083_r', 'im111_r']

units = unit_table.set_index('unit_id').loc[unitData[str(sessionId)]['unitIds'][:]]
spikes = unitData[str(sessionId)]['spikes']
highQuality = du.apply_unit_quality_filter(units, no_abnorm=False)


inRegion = du.getUnitsInRegion(units,region)
final_unit_filter = highQuality & inRegion

cols_to_add = [f'{metric}_{region}_{unitSampleName}' for unitSampleName in unitSampleNames for metric in ['previous_image_confidence', 'change_confidence']]
stim[cols_to_add] = np.nan

sp = np.zeros((final_unit_filter.sum(),spikes.shape[1],spikes.shape[2]),dtype=bool)
for i,u in enumerate(np.where(final_unit_filter)[0]):
    sp[i]=spikes[u,:,:]

flash_response = sp[:, :, respWin].mean(axis=2)

In [ ]:
pca = PCA(n_components=3)
reduced_pca = pca.fit_transform(flash_response.T)
non_omitted = ~(stim['omitted'].astype(bool).values)


In [ ]:
lda_clf = LDA(n_components=2)
non_omitted = ~(stim['omitted'].astype(bool).values)
lda_clf.fit(flash_response[:, non_omitted].T, stim[non_omitted]['image_name'])
reduced_image = lda_clf.transform(flash_response.T)

lda_change_clf = LDA()
lda_change_clf.fit(flash_response[:, non_omitted].T, stim[non_omitted]['is_change'])
reduced_change = lda_change_clf.transform(flash_response.T)

In [ ]:
reduced = np.hstack([reduced_image, reduced_change])
#reduced = reduced_pca

In [ ]:
im_id_ints = np.array([np.where(im==np.unique(im_ids))[0][0] for im in im_ids])

### Plot colored by image ID

In [ ]:
save_dir = "/Volumes/programs/mindscope/workgroups/np-behavior/VBN Manuscript/computation_of_change"

In [ ]:
# Plot colored by image ID
im_id_to_color = 0
num_to_show = 1
ims_to_show = np.zeros(len(im_ids), dtype=bool)
ims_to_show[::num_to_show] = True
fig = plt.figure()
fig.suptitle('colored_by_image')
fig.patch.set_facecolor('white')

fig.set_size_inches(8,8)
ax = fig.add_subplot(projection='3d')
ax.set_facecolor('white')

# Set the pane colors to white
ax.w_xaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
ax.w_yaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
ax.w_zaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))

# Remove grid lines
ax.grid(False)

ax.scatter(reduced[non_omitted & (im_id_ints==im_id_to_color) & ims_to_show & ~(stim['is_change']), 0], 
           reduced[non_omitted & (im_id_ints==im_id_to_color) & ims_to_show & ~(stim['is_change']), 1], 
           reduced[non_omitted & (im_id_ints==im_id_to_color) & ims_to_show & ~(stim['is_change']), 2], 
           c='teal', alpha=0.5, edgecolors='none', s=50)
ax.scatter(reduced[non_omitted & ~(im_id_ints==im_id_to_color) & ims_to_show & ~(stim['is_change']), 0], 
           reduced[non_omitted & ~(im_id_ints==im_id_to_color) & ims_to_show & ~(stim['is_change']), 1], 
           reduced[non_omitted & ~(im_id_ints==im_id_to_color) & ims_to_show & ~(stim['is_change']), 2], 
           c='gray', alpha=0.2, edgecolors='none', s=50)
ax.scatter(reduced[non_omitted & ~(im_id_ints==im_id_to_color) & (stim['is_change']) & ims_to_show, 0], 
           reduced[non_omitted & ~(im_id_ints==im_id_to_color) & (stim['is_change']) & ims_to_show, 1], 
           reduced[non_omitted & ~(im_id_ints==im_id_to_color) & (stim['is_change']) & ims_to_show, 2], 
           c='w', alpha=0.5, edgecolors='gray',facecolors='w', s=40)
ax.scatter(reduced[non_omitted & (im_id_ints==im_id_to_color) & (stim['is_change']) & ims_to_show, 0], 
           reduced[non_omitted & (im_id_ints==im_id_to_color) & (stim['is_change']) & ims_to_show, 1], 
           reduced[non_omitted & (im_id_ints==im_id_to_color) & (stim['is_change']) & ims_to_show, 2], 
           c='w', alpha=0.5, edgecolors='teal',facecolors='w', s=40)
ax.scatter(reduced[~non_omitted, 0], 
           reduced[~non_omitted, 1], 
           reduced[~non_omitted, 2], 
           c='yellow', alpha=0.1, edgecolors='none',facecolors='w', s=40)


#ax.view_init(elev=elev, azim=azim)

fig.savefig(os.path.join(save_dir, f'{sessionId}_colored_by_image_test.pdf'))

Reduce to just 3 images

In [ ]:
images_to_plot = [1,2]
im_id_to_color = images_to_plot[0]
to_color = im_id_ints == im_id_to_color
not_to_color = np.isin(im_id_ints, images_to_plot) & ~to_color

lda_clf = LDA()
non_omitted = ~(stim['omitted'].astype(bool).values)
lda_clf.fit(flash_response[:, to_color | not_to_color].T, to_color[to_color | not_to_color])
reduced_image = lda_clf.transform(flash_response.T)

lda_change_clf = LDA()
lda_change_clf.fit(flash_response[:, to_color | not_to_color].T, stim[to_color | not_to_color]['is_change'])
reduced_change = lda_change_clf.transform(flash_response.T)
change_preds = lda_change_clf.predict(flash_response.T)

In [ ]:
point_on_boundary = np.zeros(len(lda_change_clf.coef_[0]))
point_on_boundary[0] = -lda_change_clf.intercept_/lda_change_clf.coef_[0][0]
lda_change_boundary = lda_change_clf.transform(point_on_boundary.reshape(1,-1))

point_on_boundary = np.zeros(len(lda_clf.coef_[0]))
point_on_boundary[0] = -lda_clf.intercept_/lda_clf.coef_[0][0]
lda_image_boundary = lda_clf.transform(point_on_boundary.reshape(1,-1))

lda_change_boundary[0], lda_image_boundary[0]

In [ ]:
reduced = np.hstack([reduced_image, reduced_change])

In [ ]:
# find a change from the first image to the second image
eligible_changes = stim[(stim['previous_image_skipping_omitted']==np.unique(im_ids)[images_to_plot[0]])&(stim['image_name']==np.unique(im_ids)[images_to_plot[1]])]
eligible_change_inds = eligible_changes.index.values

In [ ]:
images_used = np.unique(im_ids)[images_to_plot[0]]+'_'+np.unique(im_ids)[images_to_plot[1]]
images_used

In [ ]:
# Plot colored by image ID in 2D
num_to_show = 1
ims_to_show = np.zeros(len(im_ids), dtype=bool)
ims_to_show[::num_to_show] = True

eligible_change_ind = eligible_change_inds[3]

fig, ax = plt.subplots()
fig.suptitle('colored_by_image')
fig.patch.set_facecolor('white')

fig.set_size_inches(8,8)
ax.set_facecolor('white')

# Set the pane colors to white
# ax.w_xaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
# ax.w_yaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))

# Remove grid lines
ax.grid(False)

ax.scatter(reduced[non_omitted & to_color & ims_to_show & ~(stim['is_change']), 0], 
           reduced[non_omitted & to_color & ims_to_show & ~(stim['is_change']), 1], 
           c='teal', alpha=0.1, edgecolors='none', s=50)
ax.scatter(reduced[non_omitted & not_to_color & ims_to_show & ~(stim['is_change']), 0], 
           reduced[non_omitted & not_to_color & ims_to_show & ~(stim['is_change']), 1], 
           c='gray', alpha=0.1, edgecolors='none', s=50)
ax.scatter(reduced[non_omitted & not_to_color & (stim['is_change']) & ims_to_show, 0], 
           reduced[non_omitted & not_to_color & (stim['is_change']) & ims_to_show, 1], 
           c='w', alpha=0.3, edgecolors='gray',facecolors='w', s=40)
ax.scatter(reduced[non_omitted & to_color & (stim['is_change']) & ims_to_show, 0], 
           reduced[non_omitted & to_color & (stim['is_change']) & ims_to_show, 1], 
           c='w', alpha=0.3, edgecolors='teal',facecolors='w', s=40)

ax.scatter(reduced[eligible_change_ind, 0],reduced[eligible_change_ind, 1], c='r')
ax.scatter(reduced[eligible_change_ind+1, 0],reduced[eligible_change_ind+1, 1], c='r')

for nback in range(1,4):
    ax.scatter(reduced[eligible_change_ind-nback, 0],reduced[eligible_change_ind-nback, 1], c='teal', alpha=1/nback)
#ax.scatter(reduced[eligible_change_ind-1, 0],reduced[eligible_change_ind-1, 1], c='b')


ax.axhline(lda_change_boundary[0], color='k', linestyle='--')
ax.axvline(lda_image_boundary[0], color='k', linestyle='--')
# ax.scatter(reduced[~non_omitted, 0], 
#            reduced[~non_omitted, 1], 
#            reduced[~non_omitted, 2], 
#            c='yellow', alpha=0.1, edgecolors='none',facecolors='w', s=40)


#ax.view_init(elev=elev, azim=azim)
fig.savefig(os.path.join(save_dir, f'{sessionId}_{images_used}_test.pdf'))

In [ ]:
g_image_set = pd.read_pickle("/Volumes/programs/braintv/workgroups/nc-ophys/corbettb/dev/Natural_Images_Lum_Matched_set_ophys_G_2019.05.26.pkl")


In [ ]:
from notebook_utils import plot_image

# Example usage
fig1 = plot_image(g_image_set[np.unique(im_ids)[images_to_plot[0]]][np.unique(im_ids)[images_to_plot[0]]], gamma=0.6)
fig2 = plot_image(g_image_set[np.unique(im_ids)[images_to_plot[1]]][np.unique(im_ids)[images_to_plot[1]]], gamma=0.6)

fig1.savefig(os.path.join(save_dir, np.unique(im_ids)[images_to_plot[0]] + '_test.png'), transparent=True)
fig2.savefig(os.path.join(save_dir, np.unique(im_ids)[images_to_plot[1]] + '_test.png'), transparent=True)

In [ ]:
fig, ax = plt.subplots()
ax.imshow(g_image_set[np.unique(im_ids)[images_to_plot[0]]][np.unique(im_ids)[images_to_plot[0]]], cmap='Greys_r')
ax.set_visible(False)
fig, ax = plt.subplots()
ax.imshow(g_image_set[np.unique(im_ids)[images_to_plot[1]]][np.unique(im_ids)[images_to_plot[1]]], cmap='Greys_r')

In [ ]:
# Plot colored by image ID
images_to_plot = [0, 1, 2]
im_id_to_color = images_to_plot[0]
to_color = im_id_ints == im_id_to_color
not_to_color = np.isin(im_id_ints, images_to_plot) & ~to_color

num_to_show = 1
ims_to_show = np.zeros(len(im_ids), dtype=bool)
ims_to_show[::num_to_show] = True

fig = plt.figure()
fig.suptitle('colored_by_image')
fig.patch.set_facecolor('white')

fig.set_size_inches(8,8)
ax = fig.add_subplot(projection='3d')
ax.set_facecolor('white')

# Set the pane colors to white
ax.w_xaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
ax.w_yaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
ax.w_zaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))

# Remove grid lines
ax.grid(False)

ax.scatter(reduced[non_omitted & to_color & ims_to_show & ~(stim['is_change']), 0], 
           reduced[non_omitted & to_color & ims_to_show & ~(stim['is_change']), 1], 
           reduced[non_omitted & to_color & ims_to_show & ~(stim['is_change']), 2], 
           c='teal', alpha=0.5, edgecolors='none', s=50)
ax.scatter(reduced[non_omitted & not_to_color & ims_to_show & ~(stim['is_change']), 0], 
           reduced[non_omitted & not_to_color & ims_to_show & ~(stim['is_change']), 1], 
           reduced[non_omitted & not_to_color & ims_to_show & ~(stim['is_change']), 2], 
           c='gray', alpha=0.2, edgecolors='none', s=50)
ax.scatter(reduced[non_omitted & not_to_color & (stim['is_change']) & ims_to_show, 0], 
           reduced[non_omitted & not_to_color & (stim['is_change']) & ims_to_show, 1], 
           reduced[non_omitted & not_to_color & (stim['is_change']) & ims_to_show, 2], 
           c='w', alpha=0.5, edgecolors='gray',facecolors='w', s=40)
ax.scatter(reduced[non_omitted & to_color & (stim['is_change']) & ims_to_show, 0], 
           reduced[non_omitted & to_color & (stim['is_change']) & ims_to_show, 1], 
           reduced[non_omitted & to_color & (stim['is_change']) & ims_to_show, 2], 
           c='w', alpha=0.5, edgecolors='teal',facecolors='w', s=40)
# ax.scatter(reduced[~non_omitted, 0], 
#            reduced[~non_omitted, 1], 
#            reduced[~non_omitted, 2], 
#            c='yellow', alpha=0.1, edgecolors='none',facecolors='w', s=40)


#ax.view_init(elev=elev, azim=azim)

In [ ]:
plt.figure()
plt.scatter(reduced[non_omitted & (im_id_ints==im_id_to_color) & ims_to_show, 0], 
           reduced[non_omitted & (im_id_ints==im_id_to_color) & ims_to_show, 1], 
           c='teal', alpha=0.5, edgecolors='none', s=50)

In [ ]:
elev, azim = ax.elev, ax.azim

### Plot colored by is_change

In [ ]:
# Plot colored by image ID
im_id_to_color = 4
num_to_show = 1
ims_to_show = np.zeros(len(im_ids), dtype=bool)
ims_to_show[::num_to_show] = True
fig = plt.figure()
fig.suptitle('is_change')
fig.patch.set_facecolor('white')

fig.set_size_inches(12,12)
ax = fig.add_subplot(projection='3d')
ax.set_facecolor('white')

# Set the pane colors to white
ax.w_xaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
ax.w_yaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
ax.w_zaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))

# Remove grid lines
ax.grid(False)

ax.scatter(reduced[non_omitted & (stim['is_change']) & ims_to_show, 0], 
           reduced[non_omitted & (stim['is_change']) & ims_to_show, 1], 
           reduced[non_omitted & (stim['is_change']) & ims_to_show, 2], 
           c='r', alpha=0.5, edgecolors='none', s=40)
ax.scatter(reduced[non_omitted & ~(stim['is_change']) & ims_to_show, 0], 
           reduced[non_omitted & ~(stim['is_change']) & ims_to_show, 1], 
           reduced[non_omitted & ~(stim['is_change']) & ims_to_show, 2], 
           c='gray', alpha=0.2, edgecolors='none', s=50)

ax.view_init(elev=elev, azim=azim)

fig.savefig(os.path.join(save_dir, f'{sessionId}_colored_by_change_test.pdf'))

In [ ]:
# Plot colored by is_change

fig = plt.figure()
fig.suptitle('is_change')
fig.set_size_inches(12,12)
ax = fig.add_subplot(projection='3d')
ax.scatter(reduced[:,0], reduced[:, 1], reduced[:, 2], c=stim['is_change'], alpha=0.5)

In [ ]:
# Plot colored by image ID
im_id_to_color = 4
num_to_show = 1
ims_to_show = np.zeros(len(im_ids), dtype=bool)
ims_to_show[::num_to_show] = True
fig, ax = plt.subplots()
fig.suptitle('colored_by_image')
#fig.patch.set_facecolor('white')

fig.set_size_inches(12,12)
#ax.set_facecolor('white')

# Set the pane colors to white
# ax.w_xaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
# ax.w_yaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))

# Remove grid lines
#ax.grid(False)

ax.scatter(reduced[non_omitted & (stim['is_change']) & ims_to_show, 0], 
           reduced[non_omitted & (stim['is_change']) & ims_to_show, 1], 
           c='red', alpha=0.5, edgecolors='none', s=50)
ax.scatter(reduced[non_omitted & ~(stim['is_change']) & ims_to_show, 0], 
           reduced[non_omitted & ~(stim['is_change']) & ims_to_show, 1],  
           c='gray', alpha=0.2, edgecolors='none', s=50)
#ax.view_init(elev=elev, azim=azim)

#fig.savefig(os.path.join(save_dir, f'{sessionId}_colored_by_image.pdf'))

In [ ]:
# Plot colored by image ID
im_id_to_color = 0
num_to_show = 1
ims_to_show = np.zeros(len(im_ids), dtype=bool)
ims_to_show[::num_to_show] = True
fig = plt.figure()
fig.suptitle('colored_by_image')
fig.patch.set_facecolor('white')

fig.set_size_inches(8,8)
ax = fig.add_subplot(projection='3d')
ax.set_facecolor('white')

# Set the pane colors to white
ax.w_xaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
ax.w_yaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
ax.w_zaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))

# Remove grid lines
ax.grid(False)

ax.scatter(reduced[non_omitted & (im_id_ints==im_id_to_color) & ims_to_show & ~(stim['is_change']), 0], 
           reduced[non_omitted & (im_id_ints==im_id_to_color) & ims_to_show & ~(stim['is_change']), 1], 
           reduced[non_omitted & (im_id_ints==im_id_to_color) & ims_to_show & ~(stim['is_change']), 2], 
           c='teal', alpha=0.5, edgecolors='none', s=50)
ax.scatter(reduced[non_omitted & ~(im_id_ints==im_id_to_color) & ims_to_show & ~(stim['is_change']), 0], 
           reduced[non_omitted & ~(im_id_ints==im_id_to_color) & ims_to_show & ~(stim['is_change']), 1], 
           reduced[non_omitted & ~(im_id_ints==im_id_to_color) & ims_to_show & ~(stim['is_change']), 2], 
           c='gray', alpha=0.2, edgecolors='none', s=50)
ax.scatter(reduced[non_omitted & ~(im_id_ints==im_id_to_color) & (stim['is_change']) & ims_to_show, 0], 
           reduced[non_omitted & ~(im_id_ints==im_id_to_color) & (stim['is_change']) & ims_to_show, 1], 
           reduced[non_omitted & ~(im_id_ints==im_id_to_color) & (stim['is_change']) & ims_to_show, 2], 
           c='w', alpha=0.5, edgecolors='gray',facecolors='w', s=40)
ax.scatter(reduced[non_omitted & (im_id_ints==im_id_to_color) & (stim['is_change']) & ims_to_show, 0], 
           reduced[non_omitted & (im_id_ints==im_id_to_color) & (stim['is_change']) & ims_to_show, 1], 
           reduced[non_omitted & (im_id_ints==im_id_to_color) & (stim['is_change']) & ims_to_show, 2], 
           c='w', alpha=0.5, edgecolors='teal',facecolors='w', s=40)
ax.scatter(reduced[~non_omitted, 0], 
           reduced[~non_omitted, 1], 
           reduced[~non_omitted, 2], 
           c='yellow', alpha=0.1, edgecolors='none',facecolors='w', s=40)


#ax.view_init(elev=elev, azim=azim)

fig.savefig(os.path.join(save_dir, f'{sessionId}_colored_by_image_test.pdf'))

In [ ]:
rts = stim_table['reaction_time'].values

In [ ]:
#reaction times that are faster than 100 ms should be attributed to previous flash
too_short = stim_table[stim_table['reaction_time']<0.1].index.values
too_short_rts = stim_table.loc[too_short]['reaction_time'].values
last_stim_start_diff = stim_table.loc[too_short]['start_time'].values - stim_table.loc[too_short-1]['start_time'].values

#remove weird values that are likely between sessions
good_intervals = (last_stim_start_diff>0) & (last_stim_start_diff<1)
too_short = too_short[good_intervals]
last_stim_start_diff = last_stim_start_diff[good_intervals]

#don't overwrite reaction times that already exist
no_rt = stim_table.loc[too_short-1]['reaction_time'].isna()
too_short = too_short[no_rt]
last_stim_start_diff = last_stim_start_diff[no_rt]

#find new reaction times
new_rts = stim_table.loc[too_short]['reaction_time'].values
new_rts = new_rts + last_stim_start_diff
stim_table.loc[too_short-1, 'reaction_time'] = new_rts
stim_table.loc[too_short, 'reaction_time'] = np.nan

In [ ]:
stim_table = stim_table.drop(columns='Unnamed: 0') #drop redundant column
stim_table['is_shared'] = stim_table['image_name'].isin(['im083_r', 'im111_r'])

rt_quintiles = stim_table.groupby('session_id').apply(lambda group: pd.qcut(group['reaction_time'], 5, labels=False))
stim_table['rt_quintiles'] = rt_quintiles.values

In [ ]:
stim_table.pivot_table(index='session_id', columns='rt_quintiles', values='reaction_time', aggfunc=['count', 'median'])

In [ ]:
from notebook_utils import get_session_stim_table_stats

In [ ]:
stats = stim_table.groupby('session_id').apply(get_session_stim_table_stats)